# Data for code analysis
Copy and validate users uploads before analysis

# Check uploads

In [8]:
import os
import shutil
import pandas as pd

activity_csv_path = "../../data/database/activity.csv"
if not os.path.exists(activity_csv_path):
    raise FileNotFoundError(f"Le fichier de données de l'expérimentation {activity_csv_path} n'existe pas.")

users_csv_path = "../../generated/database/users.csv"
if not os.path.exists(users_csv_path):
    raise FileNotFoundError(f"Le fichier {users_csv_path} n'existe pas. Exécutez le notebook 'notebooks/arrange data/Database data.ipynb' pour le générer.")

users = pd.read_csv(users_csv_path)[["user", "created_at", "last_update"]]
users['created_at'] = pd.to_datetime(users['created_at'], utc=True).dt.tz_convert("Europe/Brussels")
users['last_update'] = pd.to_datetime(users['last_update'], utc=True).dt.tz_convert("Europe/Brussels")

users

,user,created_at,last_update
0,11555248-3f01-4d1c-9a71-ef7caf9150fa,2026-04-16 11:12:53.421000+02:00,2026-04-16 11:12:53.421000+02:00
1,70e85b5e-92d4-498a-9079-ba881f5b3b82,2026-04-16 11:12:53.422000+02:00,2026-04-16 11:12:53.422000+02:00
2,c27240eb-52c0-4436-aa0f-e7f97a94a725,2026-04-16 11:12:56.532000+02:00,2026-04-16 11:12:56.532000+02:00
3,0d271530-be17-4538-bf04-dde3c6069b5f,2026-04-16 11:12:58.644000+02:00,2026-04-16 11:12:58.644000+02:00
4,7234f6dc-a316-4576-8453-6e10d7cf1c3d,2026-04-16 11:13:05.993000+02:00,2026-04-16 11:13:05.993000+02:00
5,84c2c4e6-1c27-4f2c-808b-9e84a7cb257e,2026-04-16 11:13:06.492000+02:00,2026-04-16 11:13:06.492000+02:00
6,bcec21aa-2246-4ee8-9f2a-3998a5cde697,2026-04-16 11:13:55.230000+02:00,2026-04-16 11:13:55.230000+02:00
7,a387c54c-b5e0-4f3c-b4aa-6f79579007a4,2026-04-16 11:14:25.132000+02:00,2026-04-16 11:14:25.132000+02:00
8,960ad9f8-2b83-4526-9cfb-1a59cec049a6,2026-04-16 11:16:16.341000+02:00,2026-04-16 11:16:16.341000+02:00
9,d37be827-99be-44be-8586-54bba37a59ce,2026-04-16 11:19:50.180000+02:00,2026-04-16 11:19:50.180000+02:00


In [9]:
uploads_path = "../../data/uploads"
directories = os.listdir(uploads_path)

for user_id in directories:
    child = os.listdir(uploads_path + "/" + user_id)
    uploads_count = len(child)

    users.loc[users["user"] == user_id, "uploads_count"] = uploads_count

    if uploads_count > 2:
        print(f"{user_id} has {uploads_count} directories instead of 2")
        for folder_name in child:
            print(f"\t{folder_name}")
        print("")


users["uploads_count"] = users["uploads_count"].fillna(0)
users["uploads_count"] = users["uploads_count"].astype(int)

# User mismatch send count

In [10]:
activity_df = pd.read_csv(activity_csv_path)[["user"]]
activity_counts = activity_df["user"].value_counts().reset_index()
activity_counts.columns = ["user", "activity_count"]

mismatch_send_df = users[users["uploads_count"] != 1]
mismatch_send_df = mismatch_send_df.merge(activity_counts, on="user", how="left")
mismatch_send_df["activity_count"] = mismatch_send_df["activity_count"].fillna(0)
mismatch_send_df.to_csv("../../generated/database/users_send_mismatch.csv", index=False)
mismatch_send_df


,user,created_at,last_update,uploads_count,activity_count
0,a387c54c-b5e0-4f3c-b4aa-6f79579007a4,2026-04-16 11:14:25.132000+02:00,2026-04-16 11:14:25.132000+02:00,0,0.0
1,d37be827-99be-44be-8586-54bba37a59ce,2026-04-16 11:19:50.180000+02:00,2026-04-16 11:19:50.180000+02:00,0,0.0


# Users with less than 1 upload

In [ ]:
less_than_1 = mismatch_send_df[mismatch_send_df["uploads_count"] < 1]
less_than_1 = less_than_1[["user", "uploads_count", "activity_count"]]
less_than_1.to_csv("../../generated/database/users_send_mismatch_less_1.csv", index=False)

# Copy & clean data

In [18]:


uploads_path = "../../data/uploads"
directories = os.listdir(uploads_path)

out_uploads_path = "../../generated/uploads"

if os.path.exists(out_uploads_path):
    print(f"Deleting {out_uploads_path}\n")
    shutil.rmtree(out_uploads_path)

os.makedirs(out_uploads_path)

for user_id in directories:
    user_uploads_path = uploads_path + "/" + user_id

    child = os.listdir(user_uploads_path)
    uploads_count = len(child)

    if uploads_count < 1:
        print(f"Ignoring {user_id} - {uploads_count} uploaded project(s)")
        continue

    out_user_uploads_path = out_uploads_path + "/" + user_id

    os.makedirs(out_user_uploads_path)
    shutil.copytree(f"{user_uploads_path}/{child[0]}", f"{out_user_uploads_path}", dirs_exist_ok=True)


    child.sort()


Deleting ../../generated/uploads



# Verified data

In [ ]:
directories = os.listdir(out_uploads_path)
invalid_directories = 0

for user_id in directories:
    child = os.listdir(out_uploads_path + "/" + user_id)
    uploads_count = len(child)

    if uploads_count >= 3 and uploads_count <= 4:
        invalid_directories = invalid_directories + 1
        print(f"{user_id} has {uploads_count} files instead of 3 or 4")
        for folder_name in child:
            print(f"\t{folder_name}")
        print("")


if invalid_directories == 0:
    print("All users have 3 or 4 files")

All users have 4 or 5 directories
